# 15 RNN / LSTM — Exercises

Test your understanding of recurrent computation, BPTT, and gating
mechanisms.

**Prerequisites.** Read [theory.md](theory.md) and work through
[first_principles.ipynb](first_principles.ipynb) before attempting these.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np

%load_ext autoreload
%autoreload 2

SEED = 42
random.seed(SEED)
rng = np.random.default_rng(SEED)

## Exercise 1 — Hand Calculation: One RNN Forward Step

Consider a vanilla RNN with $d = 2$ (input dim) and $m = 2$ (hidden dim).

**Recurrence:** $h_t = \tanh(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b_h)$

**Given:**

| Parameter | Value |
|---|---|
| $W_{xh}$ | $\begin{bmatrix} 0.5 & 0.3 \\ -0.2 & 0.4 \end{bmatrix}$ |
| $W_{hh}$ | $\begin{bmatrix} 0.1 & -0.1 \\ 0.2 & 0.3 \end{bmatrix}$ |
| $b_h$ | $\begin{bmatrix} 0.0 \\ 0.1 \end{bmatrix}$ |
| $h_0$ | $\begin{bmatrix} 0.0 \\ 0.0 \end{bmatrix}$ |
| $x_1$ | $\begin{bmatrix} 1.0 \\ 0.5 \end{bmatrix}$ |
| $x_2$ | $\begin{bmatrix} -0.5 \\ 1.0 \end{bmatrix}$ |

**Tasks (derive by hand, then verify numerically):**

1. Compute $z_1 = W_{xh}\, x_1 + W_{hh}\, h_0 + b_h$ and $h_1 = \tanh(z_1)$.
2. Compute $z_2 = W_{xh}\, x_2 + W_{hh}\, h_1 + b_h$ and $h_2 = \tanh(z_2)$.

**Step-by-step for $h_1$:**

$$W_{xh}\, x_1 = \begin{bmatrix} 0.5 \cdot 1.0 + 0.3 \cdot 0.5 \\ -0.2 \cdot 1.0 + 0.4 \cdot 0.5 \end{bmatrix}
= \begin{bmatrix} 0.65 \\ 0.0 \end{bmatrix}$$

$$W_{hh}\, h_0 = \begin{bmatrix} 0.0 \\ 0.0 \end{bmatrix}$$

$$z_1 = \begin{bmatrix} 0.65 \\ 0.0 \end{bmatrix} + \begin{bmatrix} 0.0 \\ 0.0 \end{bmatrix} + \begin{bmatrix} 0.0 \\ 0.1 \end{bmatrix}
= \begin{bmatrix} 0.65 \\ 0.1 \end{bmatrix}$$

$$h_1 = \tanh(z_1) = \begin{bmatrix} \tanh(0.65) \\ \tanh(0.1) \end{bmatrix}
\approx \begin{bmatrix} 0.5717 \\ 0.0997 \end{bmatrix}$$

**Expected results (verify to 4 d.p.):**

| Quantity | Value |
|---|---:|
| $z_1$ | $[0.6500,\; 0.1000]$ |
| $h_1$ | $[0.5717,\; 0.0997]$ |
| $z_2$ | $[-0.1028,\; 0.6299]$ |
| $h_2$ | $[-0.1024,\; 0.5581]$ |

In [ ]:
# Given values
W_xh = np.array([[0.5, 0.3], [-0.2, 0.4]])
W_hh = np.array([[0.1, -0.1], [0.2, 0.3]])
b_h = np.array([0.0, 0.1])
h_0 = np.array([0.0, 0.0])
x_1 = np.array([1.0, 0.5])
x_2 = np.array([-0.5, 1.0])

# TODO: Step 1 — compute h_1
# z_1 = W_xh @ x_1 + W_hh @ h_0 + b_h
# h_1 = np.tanh(z_1)

# TODO: Step 2 — compute h_2
# z_2 = W_xh @ x_2 + W_hh @ h_1 + b_h
# h_2 = np.tanh(z_2)

# Uncomment to verify:
# assert np.allclose(z_1, [0.65, 0.1], atol=1e-4), f'z_1 mismatch: {z_1}'
# assert np.allclose(h_1, [0.5717, 0.0997], atol=1e-4), f'h_1 mismatch: {h_1}'
# assert np.allclose(z_2, [-0.1028, 0.6299], atol=1e-4), f'z_2 mismatch: {z_2}'
# assert np.allclose(h_2, [-0.1024, 0.5581], atol=1e-4), f'h_2 mismatch: {h_2}'
# print(f'z_1 = {z_1}')
# print(f'h_1 = {h_1}')
# print(f'z_2 = {z_2}')
# print(f'h_2 = {h_2}')
# print('All hand-calculation checks passed. ✓')

## Exercise 2 — Coding: RNN Forward Pass with Gradient Check

Implement a complete vanilla RNN forward pass that processes a sequence
and returns all hidden states.

**Specification:**

$$h_t = \tanh(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b_h)$$

- Input: $X \in \mathbb{R}^{T \times d}$ (one sequence, $T$ time steps)
- Initial state: $h_0 = \mathbf{0}$
- Output: $H \in \mathbb{R}^{T \times m}$ (all hidden states)

**Tasks:**

1. Implement `rnn_forward(X, W_xh, W_hh, b_h)` that returns all hidden states.
2. Verify with a deterministic check on fixed inputs.
3. Verify the gradient of $\sum h_T$ w.r.t. $W_{xh}$ using finite differences.

**Deterministic check:** Output shapes and specific numerical values must match.

In [ ]:
def rnn_forward(X, W_xh, W_hh, b_h):
    """Vanilla RNN forward pass.

    Args:
        X: (T, d) input sequence
        W_xh: (m, d) input-to-hidden weights
        W_hh: (m, m) hidden-to-hidden weights
        b_h: (m,) bias

    Returns:
        H: (T, m) all hidden states
    """
    # TODO: implement
    # T, d = X.shape
    # m = W_hh.shape[0]
    # h = np.zeros(m)
    # H = []
    # for t in range(T):
    #     h = np.tanh(W_xh @ X[t] + W_hh @ h + b_h)
    #     H.append(h.copy())
    # return np.array(H)
    pass

In [ ]:
# Deterministic test
rng_ex2 = np.random.default_rng(99)
T_ex2, d_ex2, m_ex2 = 5, 2, 3
W_xh_ex2 = rng_ex2.normal(size=(m_ex2, d_ex2)) * 0.5
W_hh_ex2 = rng_ex2.normal(size=(m_ex2, m_ex2)) * 0.3
b_h_ex2 = np.zeros(m_ex2)
X_ex2 = rng_ex2.normal(size=(T_ex2, d_ex2))

# TODO: uncomment after implementing rnn_forward
# H_ex2 = rnn_forward(X_ex2, W_xh_ex2, W_hh_ex2, b_h_ex2)
# assert H_ex2.shape == (5, 3), f'Shape mismatch: {H_ex2.shape}'
# assert np.all(np.abs(H_ex2) <= 1.0), 'tanh outputs must be in [-1, 1]'
#
# # Gradient check: d(sum(h_T)) / d(W_xh) via finite differences
# eps = 1e-5
# grad_num = np.zeros_like(W_xh_ex2)
# for i in range(m_ex2):
#     for j in range(d_ex2):
#         W_xh_ex2[i, j] += eps
#         H_plus = rnn_forward(X_ex2, W_xh_ex2, W_hh_ex2, b_h_ex2)
#         W_xh_ex2[i, j] -= 2 * eps
#         H_minus = rnn_forward(X_ex2, W_xh_ex2, W_hh_ex2, b_h_ex2)
#         W_xh_ex2[i, j] += eps  # restore
#         grad_num[i, j] = (H_plus[-1].sum() - H_minus[-1].sum()) / (2 * eps)
#
# print(f'Numerical gradient of sum(h_T) w.r.t. W_xh:')
# print(grad_num)
# print(f'\nOutput shape: {H_ex2.shape} ✓')
# print(f'All outputs in [-1, 1]: ✓')
# print('RNN forward pass implementation verified. ✓')

## Exercise 3 — Conceptual: Why Does LSTM's Cell State Act as a Gradient Highway?

**Questions:**

1. **Vanilla RNN gradient path.** In a vanilla RNN, the gradient from step $T$
   to step $t$ passes through the product:

   $$\prod_{s=t+1}^{T} \text{diag}(\tanh'(z_s))\, W_{hh}$$

   Each factor includes both an element-wise nonlinearity ($\tanh'$) **and** the
   recurrent weight matrix $W_{hh}$. Explain why this causes vanishing gradients
   even when $\|W_{hh}\|$ is moderate.

2. **LSTM cell state gradient.** In an LSTM, the gradient of $c_T$ w.r.t. $c_t$ is:

   $$\frac{\partial c_T}{\partial c_t} = \prod_{s=t+1}^{T} f_s$$

   where $f_s$ is the forget gate at step $s$. Explain why this is a "gradient
   highway" — why is this product better behaved than the vanilla RNN product?

3. **When LSTM still fails.** Give a concrete scenario where even an LSTM cannot
   learn a long-range dependency. What architectural modification addresses this?

4. **Forget gate bias initialization.** Why is $b_f$ often initialized to a
   positive value (e.g., 1.0)? What happens if it starts at 0?

In [ ]:
# Numerical illustration: compare gradient products for RNN vs LSTM

T_steps = 50
m_dim = 8
rng_ex3 = np.random.default_rng(42)

# --- Vanilla RNN: product of diag(tanh'(z)) @ W_hh ---
W_hh_sim = rng_ex3.normal(size=(m_dim, m_dim)) * 0.3
rnn_grad_product = np.eye(m_dim)
rnn_norms = [np.linalg.norm(rnn_grad_product)]

for s in range(T_steps):
    # Simulate tanh derivative at some random pre-activation
    z_sim = rng_ex3.normal(size=m_dim)
    tanh_deriv = 1 - np.tanh(z_sim) ** 2  # values in (0, 1]
    jacobian = np.diag(tanh_deriv) @ W_hh_sim
    rnn_grad_product = jacobian @ rnn_grad_product
    rnn_norms.append(np.linalg.norm(rnn_grad_product))

# --- LSTM: product of forget gates ---
lstm_grad_product = np.ones(m_dim)
lstm_norms = [np.linalg.norm(lstm_grad_product)]

for s in range(T_steps):
    # Simulate forget gate near 0.95 (learned to remember)
    f_sim = 0.90 + 0.09 * rng_ex3.random(m_dim)  # values in [0.90, 0.99]
    lstm_grad_product = f_sim * lstm_grad_product
    lstm_norms.append(np.linalg.norm(lstm_grad_product))

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogy(rnn_norms, 'o-', color='crimson', markersize=3, label='Vanilla RNN (Jacobian product)')
ax.semilogy(lstm_norms, 's-', color='steelblue', markersize=3, label='LSTM (forget gate product)')
ax.set_xlabel('Number of backward steps')
ax.set_ylabel('Gradient product norm (log scale)')
ax.set_title('Gradient Highway: LSTM forget gates vs RNN Jacobian products')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'After {T_steps} steps:')
print(f'  RNN gradient product norm:  {rnn_norms[-1]:.2e}')
print(f'  LSTM gradient product norm: {lstm_norms[-1]:.2e}')
print(f'  Ratio (LSTM/RNN): {lstm_norms[-1] / (rnn_norms[-1] + 1e-30):.2e}')
print()
print('The LSTM cell state provides a much more stable gradient path.')
print('Key difference: the LSTM path involves only element-wise products')
print('of forget gates (scalars in [0,1]), while the RNN path involves')
print('full matrix multiplications that can compound shrinkage.')

# TODO: Answer the conceptual questions above
# Hint for Q1: max(tanh'(z)) = 1 at z=0, but typical values are << 1.
#   Combined with W_hh multiplication, the product shrinks exponentially.
# Hint for Q2: forget gate product is element-wise, no matrix multiplication.
#   When f ≈ 1, the product stays near 1.
# Hint for Q3: T > 500+, or when the model cannot learn which information to keep.
#   Attention mechanisms (Transformer) address this.
# Hint for Q4: sigmoid(0) = 0.5, so f ≈ 0.5 → information halved each step.
#   sigmoid(1) ≈ 0.73 → much better initial preservation.